# Checando conexão do Microfone:

In [1]:
!pwd

/home/rolds/KWS


In [2]:
!arecord -l

**** List of CAPTURE Hardware Devices ****
card 0: SoloCast [HyperX SoloCast], device 0: USB Audio [USB Audio]
  Subdevices: 1/1
  Subdevice #0: subdevice #0


In [3]:
import pyaudio

index = 0

p = pyaudio.PyAudio()
info = p.get_host_api_info_by_index(index)
numdevices = info.get('deviceCount')

print("Dispositivos de áudio encontrados:")
for i in range(0, numdevices):
    device_info = p.get_device_info_by_host_api_device_index(index, i)
    if device_info.get('maxInputChannels') > 0:
        print(f"  Índice de Entrada {i}: {device_info.get('name')}")

p.terminate()

Dispositivos de áudio encontrados:
  Índice de Entrada 0: HyperX SoloCast: USB Audio (hw:0,0)
  Índice de Entrada 1: sysdefault
  Índice de Entrada 2: spdif
  Índice de Entrada 3: default


ALSA lib pcm_dmix.c:1000:(snd_pcm_dmix_open) unable to open slave
ALSA lib pcm.c:2722:(snd_pcm_open_noupdate) Unknown PCM cards.pcm.rear
ALSA lib pcm.c:2722:(snd_pcm_open_noupdate) Unknown PCM cards.pcm.center_lfe
ALSA lib pcm.c:2722:(snd_pcm_open_noupdate) Unknown PCM cards.pcm.side
ALSA lib confmisc.c:1377:(snd_func_refer) Unable to find definition 'cards.0.pcm.hdmi.0:CARD=0,AES0=4,AES1=130,AES2=0,AES3=2'
ALSA lib conf.c:5205:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5728:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2722:(snd_pcm_open_noupdate) Unknown PCM hdmi
ALSA lib confmisc.c:1377:(snd_func_refer) Unable to find definition 'cards.0.pcm.hdmi.0:CARD=0,AES0=4,AES1=130,AES2=0,AES3=2'
ALSA lib conf.c:5205:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5728:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2

In [4]:
import pyaudio
import wave
import numpy as np
print(np.__version__)
from scipy.signal import resample_poly

2.3.5


In [5]:
FORMAT = pyaudio.paInt16
CHANNELS = 1
RATE = 48000
TARGET_RATE = 16000

CHUNK = 1024
RECORD_SECONDS = 10

DEVICE_INDEX = 0   # replace this with your detected USB mic's index
WAVE_OUTPUT_FILENAME = "output.wav"

In [6]:
audio = pyaudio.PyAudio()

stream = audio.open(
    format=FORMAT,
    channels=CHANNELS,
    rate=RATE,
    input=True,
    input_device_index=DEVICE_INDEX,
    frames_per_buffer=CHUNK
)

ALSA lib pcm_dmix.c:1000:(snd_pcm_dmix_open) unable to open slave
ALSA lib pcm.c:2722:(snd_pcm_open_noupdate) Unknown PCM cards.pcm.rear
ALSA lib pcm.c:2722:(snd_pcm_open_noupdate) Unknown PCM cards.pcm.center_lfe
ALSA lib pcm.c:2722:(snd_pcm_open_noupdate) Unknown PCM cards.pcm.side
ALSA lib confmisc.c:1377:(snd_func_refer) Unable to find definition 'cards.0.pcm.hdmi.0:CARD=0,AES0=4,AES1=130,AES2=0,AES3=2'
ALSA lib conf.c:5205:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5728:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2722:(snd_pcm_open_noupdate) Unknown PCM hdmi
ALSA lib confmisc.c:1377:(snd_func_refer) Unable to find definition 'cards.0.pcm.hdmi.0:CARD=0,AES0=4,AES1=130,AES2=0,AES3=2'
ALSA lib conf.c:5205:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5728:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2

In [7]:
print("Recording...")

frames = []

for i in range(0, int(RATE / CHUNK * RECORD_SECONDS)):
    data = stream.read(CHUNK, exception_on_overflow=False)
    frames.append(data)

print("Finished recording.")

stream.stop_stream()
stream.close()
audio.terminate()

audio_data = np.frombuffer(b''.join(frames), dtype=np.int16)

# Calcula os fatores para a reamostragem (de 48k para 16k é 1/3)
resampled_data = resample_poly(audio_data, TARGET_RATE, RATE)

# Converte de volta para int16
resampled_data = resampled_data.astype(np.int16)

wf = wave.open(WAVE_OUTPUT_FILENAME, 'wb')

# Define os parâmetros do ÁUDIO REAMOSTRADO
wf.setnchannels(CHANNELS)
wf.setsampwidth(audio.get_sample_size(FORMAT))
wf.setframerate(TARGET_RATE) # <-- MUDANÇA 1: Usar a taxa de amostragem de destino

# Escreve os dados REAMOSTRADOS no arquivo
# .tobytes() converte o array NumPy de volta para bytes
wf.writeframes(resampled_data.tobytes()) # <-- MUDANÇA 2: Usar os dados reamostrados

wf.close()

Recording...
Finished recording.


In [8]:
!ls

output.wav  venv


In [9]:
import IPython.display as ipd
ipd.Audio("output.wav")